> **Nur in Google Colab noetig:** Die folgende Zelle klont das Repository und macht die Bibliotheksdateien (, ) verfuegbar. Lokal oder in JupyterLite kann sie uebersprungen werden.

In [ ]:
import os, sys, importlib.util
if not os.path.exists("FEM"):
    !git clone --depth=1 -q https://github.com/Boscij/FEM.git
if "FEM/content" not in sys.path:
    sys.path.insert(0, "FEM/content")

# ipympl wird fuer das interaktive Widget benoetigt
# Pruefen ohne Import (Import wuerde matplotlib-Backend vorzeitig setzen)
if importlib.util.find_spec("ipympl") is None:
    !{sys.executable} -m pip install -q ipympl
    print("ipympl installiert — Laufzeit wird automatisch neu gestartet...")
    os.kill(os.getpid(), 9)


In [ ]:
import numpy as np
from fem_core import assemble_K, solve_system
from fem_post import postprocessing, plot_results

Gibt dir die Möglichkeit in colab eigene Widgets zu nutzen

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

# Pre-Processing - Modelleingabe GUI

Beim Erstellen eines FE-Modells muessen Knotenkoordinaten, Elemente, Querschnitte, Randbedingungen und Lasten definiert werden.
Dieses Notebook stellt dafuer eine **grafische Eingabeoberflaeche** bereit:
Das Fachwerk wird per Maus gezeichnet, und der Output ist direkt als Python-Code ausgegeben,
der in den Input-Block von  kopiert werden kann.

| Modus | Linksklick | Rechtsklick |
|---|---|---|
| **Knoten** | Knoten setzen (rastet auf 100 mm-Raster) | Knoten loeschen |
| **Stab** | 1. Klick: Startknoten waehlen, 2. Klick: Endknoten | Stab loeschen |
| **Lager** | Knoten anklicken, Randbedingungen in Sidebar setzen | - |
| **Last** | Knoten anklicken, Kraefte in Sidebar eintragen | - |

> Der Code der GUI ist **nicht Teil der Vorlesung**.
> Wer sich dafuer interessiert, kann ihn in  nachlesen.

In [ ]:
%matplotlib widget

from fem_pre import show_gui
show_gui()

---

## Modell

Klicke im GUI auf **Code generieren** und kopiere den erzeugten Code in die Zelle unten.
Einheiten: Koordinaten in [mm], Flaechen in [mm^2], E in [MPa], Kraefte in [N].

In [ ]:
# Modell aus Pre-Processing einfuegen (Code generieren -> kopieren)
nodal_coordinates = np.array([
    [   0.0,    0.0],   # Knoten 1
    [1000.0,    0.0],   # Knoten 2
    [1000.0, 1000.0],   # Knoten 3
    [2000.0, 1000.0],   # Knoten 4
])

elements = [
    [0, 1, "s1"],
    [0, 2, "s2"],
    [1, 2, "s3"],
    [1, 3, "s4"],
    [2, 3, "s5"],
]

materials = {
    "s1": [210000.0],
    "s2": [210000.0],
    "s3": [210000.0],
    "s4": [210000.0],
    "s5": [210000.0],
}

sections = {
    "s1": [15.00, "s1"],
    "s2": [28.28, "s2"],
    "s3": [10.00, "s3"],
    "s4": [56.56, "s4"],
    "s5": [10.00, "s5"],
}

constraints = [
    [0, 0, 0.0],
    [0, 1, 0.0],
    [1, 1, 0.0],
]

loads = [
    [3, 1, -1000.0],
]

## Berechnung

In [ ]:
K           = assemble_K(nodal_coordinates, elements, sections, materials)
U, F, fixed = solve_system(K, constraints, loads)
eps, sig, N = postprocessing(U, nodal_coordinates, elements, sections, materials)

## Ergebnisse

In [ ]:
print('Verschiebungen und Kraefte:')
print(f"  {'DOF':>3}  {'U [mm]':>16}  {'F [N]':>12}")
print('  ' + chr(9472) * 36)
for k in range(len(U)):
    print(f'  {k+1:>3}  {U[k]:>+16.6e}  {F[k]:>+12.4f}')

print('
Stabkraefte:')
print(f"  {'Stab':>4}  {'eps [-]':>14}  {'sig [MPa]':>10}  {'N [N]':>10}")
print('  ' + chr(9472) * 46)
for e in range(len(elements)):
    print(f'  {e+1:>4}  {eps[e]:>14.6e}  {sig[e]:>10.4f}  {N[e]:>10.4f}')

## Visualisierung

In [ ]:
plot_results(nodal_coordinates, elements, constraints, loads, U, sig, scale=200)